# Credit data simulation with BrainModelKit

This notebook shows how to generate synthetic credit data from the installed package and prepare it for Spark ML training.

## Installation

Run this once in a new notebook environment, then restart the kernel if prompted.

In [ ]:
%pip install "BrainModelKit[pyspark]"

## Generate synthetic credit data

In [ ]:
from brainmodelkit.pyspark import simulate_credit_data

credit_df, numeric_feature_columns = simulate_credit_data(
    row_count=10_000,
    feature_count=20,
    max_days=30,
    seed=42,
)

credit_df.show(10, truncate=False)
credit_df.printSchema()

## Prepare features for model training

The categorical columns are indexed and one-hot encoded. They are then combined with the generated numeric columns in a Spark ML feature vector.

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import OneHotEncoder, StringIndexer, VectorAssembler

categorical_columns = ["status", "industry_section"]
indexed_columns = [f"{column}_index" for column in categorical_columns]
encoded_columns = [f"{column}_encoded" for column in categorical_columns]

indexers = [
    StringIndexer(
        inputCol=input_column,
        outputCol=output_column,
        handleInvalid="keep",
    )
    for input_column, output_column in zip(
        categorical_columns, indexed_columns, strict=True
    )
]
encoder = OneHotEncoder(
    inputCols=indexed_columns,
    outputCols=encoded_columns,
    handleInvalid="keep",
)
assembler = VectorAssembler(
    inputCols=[*numeric_feature_columns, *encoded_columns],
    outputCol="features",
)
pipeline = Pipeline(stages=[*indexers, encoder, assembler])
prepared_df = pipeline.fit(credit_df).transform(credit_df)
model_data = prepared_df.select("features", "default_flag")
train_data, test_data = model_data.randomSplit([0.8, 0.2], seed=42)

print(f"Training rows: {train_data.count():,}")
print(f"Test rows: {test_data.count():,}")
model_data.show(5, truncate=False)

## Check default rates

In [ ]:
import pyspark.sql.functions as F

(
    credit_df.groupBy("industry_section", "status")
    .agg(
        F.count("*").alias("row_count"),
        F.avg("default_flag").alias("default_rate"),
    )
    .orderBy("industry_section", "status")
    .show(200, truncate=False)
)

## Optional export

In [ ]:
# model_data.write.mode("overwrite").parquet("synthetic_credit_training_data")